In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import os

# ==========================================
# MILESTONE 2: Dataset Collection & EDA
# ==========================================
print("--- Starting Milestone 2 & 3: Data Preparation ---")
# For demonstration, we generate a synthetic dataset.
# In a real scenario, replace this with pd.read_csv('your_actual_dataset.csv')
data = {
    "text": [
        "I feel totally prepared and calm for the math exam.",
        "I am a bit nervous about the physics test tomorrow.",
        "I am absolutely terrified, my heart is racing and I can't sleep before this exam!",
        "Studying is going well, I think I will pass.",
        "I have butterflies in my stomach, somewhat worried.",
        "I am having a panic attack thinking about the finals, I feel sick.",
        "Ready to ace this test, feeling confident.",
        "Not sure if I studied enough, feeling slightly anxious.",
        "I am so stressed I am crying, I will definitely fail."
    ],
    "label_text": ["Low", "Moderate", "High", "Low", "Moderate", "High", "Low", "Moderate", "High"]
}
df = pd.DataFrame(data)

# ==========================================
# MILESTONE 3: Data Preprocessing & Label Mapping
# ==========================================
# Map labels to integers: 0 = Low, 1 = Moderate, 2 = High
label_map = {"Low": 0, "Moderate": 1, "High": 2}
df['label'] = df['label_text'].map(label_map)

# Split the dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# Custom PyTorch Dataset
class AnxietyDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = AnxietyDataset(train_encodings, train_labels)
val_dataset = AnxietyDataset(val_encodings, val_labels)

# ==========================================
# MILESTONE 4: BERT Model Selection & Training
# ==========================================
print("--- Starting Milestone 4: BERT Model Training ---")
# Load pre-trained BERT with a sequence classification head (3 labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Define training arguments (Kept minimal for quick testing/demo)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              # Increase for real training
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=5,
    evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train the model
trainer.train()

# ==========================================
# MILESTONE 5: Model Evaluation & Saving
# ==========================================
print("--- Starting Milestone 5: Evaluation & Saving ---")
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# Save the trained model and tokenizer for the FastAPI backend
save_directory = "./bert_anxiety_model"
os.makedirs(save_directory, exist_ok=True)
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer successfully saved to {save_directory}")
print("You can now run 'uvicorn api:app --reload' to start the backend!")